### Spam Detection using Text Mining and Naive Bayes Classifier

- Each SMS message can be a spam or a ham (legitimate).

In [ ]:
# Import useful libararies used for data management
import pandas as pd
import numpy as np

# load dataset 'Spam.csv', using 'python' as engine
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html
dataset = pd.read_csv("Spam.csv", encoding = "ISO-8859-1")

In [ ]:
#https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.set_option.html
# set max_colwidth 500
pd.set_option('display.max_colwidth', 500)  

dataset.head()

In [ ]:
dataset['Class'].value_counts()

In [ ]:
# convert the 'label' column into a numeric variable; 'ham' as 0, 'spam' as 1
# Alternatively, you can do one-hot-encoding, and drop the attribute Class_ham.
dataset['Label'] = dataset['Class'].map({'ham':0, 'spam':1})

In [ ]:
dataset.head()

In [ ]:
# Now let's define X and y 

X = dataset['Text']
y = dataset['Label']

In [ ]:
# show the dimension of the X
#https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.shape.html
X.shape

In [ ]:
# Now let's do train/test split on the X and y data (70% train, 30% test)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3 , random_state=1)

In [ ]:
# show the dimension of X_train
# https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.shape.html
X_train.shape

In [ ]:
# show the dimension of X_test
# https://docs.scipy.org/doc/numpy/reference/generated/numpy.ndarray.shape.html

X_test.shape

In [ ]:
# show the number of 'ham' and 'spam' in the training set
# https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html
y_train.value_counts()

In [ ]:
# Import the CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer

# Now we are ready to vectorize the data
# first, instantiate the vectorizer
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
# When building the vocabulary ignore terms that have a document frequency strictly lower than the min_df (proportion), or higher than max_df
vectorizer = CountVectorizer(encoding='utf-8',stop_words='english',min_df=0.02, max_df=0.5)

#### TODO
- Please try TfidfVectorizer later to see the model performance (https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- You need to use "from sklearn.feature_extraction.text import TfidfVectorizer" to import the TfidfVectorizer first

In [ ]:
# Learn the vocabulary dictionary and return term-document matrix.
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer.fit_transform
X_train_vec = vectorizer.fit_transform(X_train)

In [ ]:
# print the terms
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer.get_feature_names
vectorizer.get_feature_names_out()

In [ ]:
print(X_train_vec)

In [ ]:
# Now let's transform the test data without fitting (fit only on training)! 
# https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html#sklearn.feature_extraction.text.CountVectorizer.transform
X_test_vec = vectorizer.transform(X_test)
print(X_test_vec)

In [ ]:
#Import Multinomial Naive Bayes model from sklearn
from sklearn.naive_bayes import MultinomialNB

# Create a Multinomial Naive Bayes Classifier, which is frequently used in Tf-idf
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html
mnb = MultinomialNB()
# train the model using training set 
mnb.fit(X_train_vec, y_train)

In [ ]:
# Empirical log probability of features given a class 1 (spam)
coeff_df = pd.DataFrame(mnb.feature_log_prob_[1,:].flatten(), vectorizer.get_feature_names_out(), columns=['Log Probability'])  
coeff_df = coeff_df.sort_values(by=['Log Probability'], ascending=False)

# display the 50 most informative features (indicate spam)
coeff_df.head(50)

In [ ]:
# Make class prediction for test set
# y_pred_class is the binary label
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB.predict
y_pred_class = mnb.predict(X_test_vec)

# y_pred_prob is the probability estimate
# https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html#sklearn.naive_bayes.MultinomialNB.predict_proba
y_pred_prob = mnb.predict_proba(X_test_vec)

In [ ]:
y_pred_class

In [ ]:
y_pred_prob

In [ ]:
# import libararies for evaluation measures
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

In [ ]:
print("Accuracy:",accuracy_score(y_test, y_pred_class, normalize=True, sample_weight=None))
print("Confusion Matrix:", confusion_matrix(y_test, y_pred_class))
print("Classification Report:",classification_report(y_test, y_pred_class))

In [ ]:
# Now let's see the false positive messages

X_test[(y_test == 0) & (y_pred_class==1)]

In [ ]:
# Now let's see the false negative messages

X_test[(y_test == 1) & (y_pred_class==0)]

In [ ]:
mnb.classes_

In [ ]:
from sklearn.metrics import roc_curve
from sklearn.metrics import auc

# y_test is the true label, pred_prob[:,1] measures the predicted probability of belonging to class 1 (which is defined as positive class)
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob[:,1], pos_label=1)
#fpr, tpr, thresholds = roc_curve(y_test, pred_val2, pos_label=1)

In [ ]:
# Aread under ROC curve
roc_auc=auc(fpr,tpr)
roc_auc

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure()
lw = 2
plt.plot(fpr, tpr, marker='o',color='darkorange',
         lw=lw, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver operating characteristic')
plt.legend(loc="lower right")
plt.show()